# Elliptic Graph Structural Study

This notebook runs and summarizes directed graph EDA for the Elliptic dataset.

**Outputs generated by script:**
- Node degree metrics (`degree`, `in_degree`, `out_degree`, `out_in_ratio`)
- Class-wise summary stats for licit / illicit / unknown
- Edge class mixing and illicit edge probabilities
- Time-step component and distance summaries (1–49)
- Local clustering per node and average clustering by class
- Assortativity summary and diagnostic plots

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
SCRIPT_PATH = ROOT / "scripts" / "graph_analysis.py"
DATA_DIR = ROOT / "Dataset" / "Dataset" / "raw" / "elliptic_bitcoin_dataset"
OUT_DIR = ROOT / "outputs" / "graph_eda"
TABLES = OUT_DIR / "tables"
PLOTS = OUT_DIR / "plots"

SCRIPT_PATH, DATA_DIR, OUT_DIR

(WindowsPath('c:/Users/ashis/Desktop/Finance Projects/Deep Learning project 2026/scripts/graph_analysis.py'),
 WindowsPath('c:/Users/ashis/Desktop/Finance Projects/Deep Learning project 2026/Dataset/Dataset/raw/elliptic_bitcoin_dataset'),
 WindowsPath('c:/Users/ashis/Desktop/Finance Projects/Deep Learning project 2026/outputs/graph_eda'))

In [ ]:
run_analysis = True

if run_analysis:
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--data-dir", str(DATA_DIR),
        "--out-dir", str(OUT_DIR),
    ]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, capture_output=True, text=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Graph analysis failed with exit code {completed.returncode}")

print("Done. Outputs in:", OUT_DIR)

In [ ]:
degree_summary = pd.read_csv(TABLES / "class_degree_summary.csv")
assort = pd.read_csv(TABLES / "assortativity_summary.csv")
illicit_probs = pd.read_csv(TABLES / "illicit_edge_probabilities.csv")
component_distance = pd.read_csv(TABLES / "timestamp_component_and_distance_summary.csv")
clust_avg = pd.read_csv(TABLES / "average_clustering_by_class.csv")

print("Degree summary (first rows):")
display(degree_summary.head(10))

print("Assortativity:")
display(assort)

print("Illicit edge probabilities:")
display(illicit_probs)

print("Timestamp summary (first 10):")
display(component_distance.head(10))

print("Average clustering by class:")
display(clust_avg)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

component_distance.plot(
    x="time_step",
    y=["weak_component_count", "strong_component_count"],
    ax=axes[0],
    marker="o",
    title="Connected Components by Time Step",
)
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Count")

component_distance.plot(
    x="time_step",
    y=["avg_shortest_path_lcc_undirected", "mean_illicit_illicit_distance_lcc", "mean_random_distance_lcc"],
    ax=axes[1],
    marker="o",
    title="Distance Metrics by Time Step",
)
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Distance")

plt.tight_layout()
plt.show()

from IPython.display import Image, display

display(Image(filename=str(PLOTS / "degree_histogram_log.png")))
display(Image(filename=str(PLOTS / "degree_ccdf_by_class.png")))
display(Image(filename=str(PLOTS / "in_vs_out_scatter_log.png")))

## Auto-generate Checkpoint Summary

This section creates a concise markdown report using the computed graph analysis outputs and writes it to `outputs/graph_eda/report/checkpoint_graph_summary.md`.

### Metrics Explained (with Definitions)

Let the directed transaction graph be $G=(V,E)$, where nodes are wallets/transactions and directed edges are money flow links.

- $N = |V|$: total number of nodes.
- $k_i^{in}$: in-degree of node $i$ (number of incoming edges).
- $k_i^{out}$: out-degree of node $i$ (number of outgoing edges).
- $k_i = k_i^{in}+k_i^{out}$: total degree.
- Out/In ratio for node $i$:
  $$
  r_i = \frac{k_i^{out}}{k_i^{in}+\varepsilon}
  $$
  where a small $\varepsilon>0$ avoids division by zero.

#### Class-wise Degree Statistics

For each class group $c\in\{\text{illicit},\text{licit},\text{unknown}\}$ and metric $x_i$ (e.g., degree):

- Mean:
  $$
  \mu_c = \frac{1}{|V_c|}\sum_{i\in V_c} x_i
  $$
- Standard deviation:
  $$
  \sigma_c = \sqrt{\frac{1}{|V_c|-1}\sum_{i\in V_c}(x_i-\mu_c)^2}
  $$
- Median: 50th percentile of $\{x_i: i\in V_c\}$.

#### Edge Mixing and Illicit Edge Probabilities

Let $C(u)$ be the class of source node $u$, and $C(v)$ the class of target node $v$.

- Edge mixing matrix entry:
  $$
  M_{ab} = |\{(u,v)\in E: C(u)=a,\ C(v)=b\}|
  $$
- Conditional probability of target class $b$ given source class $a$:
  $$
  P(a\rightarrow b) = \frac{M_{ab}}{\sum_{b'} M_{ab'}}
  $$

So $P(\text{illicit}\rightarrow\text{illicit})$ and $P(\text{illicit}\rightarrow\text{licit})$ quantify where illicit-origin edges go.

#### Temporal Connectivity (per time step)

For each time step subgraph $G_t$:

- Weakly connected component count: number of connected components in the undirected version of $G_t$.
- Strongly connected component count: number of SCCs in the directed $G_t$.
- Giant component share:
  $$
  \text{GiantShare}_t = \frac{|V_t^{\max}|}{|V_t|}
  $$
  where $V_t^{\max}$ is the largest weakly connected component.

#### Distance Metrics in the Largest Connected Component (LCC)

Distances are shortest-path hop counts on the undirected LCC of each time step.

- Average shortest path length:
  $$
  \bar d_t = \frac{1}{|P_t|}\sum_{(u,v)\in P_t} d_t(u,v)
  $$
  where $P_t$ is the set of connected node pairs in the LCC.
- Mean illicit-illicit distance: average $d_t(u,v)$ over pairs with both nodes illicit.
- Mean random distance: average over uniformly sampled random node pairs in the LCC.

#### Local Clustering

For node $i$ with neighbor set $N(i)$ and $k_i=|N(i)|$:

- Undirected local clustering coefficient:
  $$
  C_i^{und} = \frac{2T_i}{k_i(k_i-1)}
  $$
  where $T_i$ is the number of triangles touching $i$.

Average clustering by class is:
$$
\bar C_c = \frac{1}{|V_c|}\sum_{i\in V_c} C_i
$$

#### Assortativity

Label assortativity measures homophily by class labels across edges (Newman assortativity):
$$
r = \frac{\sum_a e_{aa} - \sum_a a_a b_a}{1 - \sum_a a_a b_a}
$$
where $e_{ab}$ is the normalized edge fraction from class $a$ to class $b$, and $a_a, b_a$ are row/column marginals of $e$.

- $r>0$: same-label nodes connect more than expected (homophily).
- $r\approx 0$: near-random mixing by label.
- $r<0$: disassortative mixing (cross-label preference).

In [3]:
from pathlib import Path
from datetime import date
from IPython.display import Markdown, display
import numpy as np
import pandas as pd

report_dir = OUT_DIR / "report"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "checkpoint_graph_summary.md"

node_metrics = pd.read_csv(TABLES / "node_degree_metrics.csv")
class_summary = pd.read_csv(TABLES / "class_degree_summary.csv")
assort = pd.read_csv(TABLES / "assortativity_summary.csv")
illicit_probs = pd.read_csv(TABLES / "illicit_edge_probabilities.csv")
comp_dist = pd.read_csv(TABLES / "timestamp_component_and_distance_summary.csv")
clust_avg = pd.read_csv(TABLES / "average_clustering_by_class.csv")


def pick_metric_table(df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    out = df[df["metric"] == metric_name].copy()
    return out[["class_group", "min", "max", "mean", "std", "median", "count"]].sort_values("class_group")


def safe_float(v, nd=4):
    if pd.isna(v):
        return "NA"
    return f"{float(v):.{nd}f}"


def df_to_pipe_markdown(df: pd.DataFrame, decimals: int = 4) -> str:
    if df.empty:
        return "(no rows)"
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].map(lambda x: "NA" if pd.isna(x) else f"{x:.{decimals}f}")
    headers = "| " + " | ".join(out.columns.astype(str).tolist()) + " |"
    divider = "| " + " | ".join(["---"] * len(out.columns)) + " |"
    rows = ["| " + " | ".join(map(str, row)) + " |" for row in out.values.tolist()]
    return "\n".join([headers, divider] + rows)


n_nodes = len(node_metrics)
n_illicit = int((node_metrics["class_group"] == "illicit").sum())
n_licit = int((node_metrics["class_group"] == "licit").sum())
n_unknown = int((node_metrics["class_group"] == "unknown").sum())

p_i_i = illicit_probs.loc[illicit_probs["metric"] == "P(illicit->illicit)", "value"]
p_i_l = illicit_probs.loc[illicit_probs["metric"] == "P(illicit->licit)", "value"]

assort_val = assort.loc[assort["metric"] == "label_attribute_assortativity", "value"]

avg_weak = comp_dist["weak_component_count"].mean()
avg_strong = comp_dist["strong_component_count"].mean()
avg_giant_share = comp_dist["giant_component_share"].mean()
avg_spl = comp_dist["avg_shortest_path_lcc_undirected"].mean()
avg_illicit_dist = comp_dist["mean_illicit_illicit_distance_lcc"].mean()
avg_random_dist = comp_dist["mean_random_distance_lcc"].mean()

avg_clust_by_class = clust_avg.copy()
for col in ["local_clustering_directed", "local_clustering_undirected"]:
    avg_clust_by_class[col] = avg_clust_by_class[col].apply(lambda x: float(x) if pd.notna(x) else np.nan)

report_lines = []
report_lines.append("# Graph Structural Analysis Summary")
report_lines.append("")
report_lines.append(f"Generated on: {date.today().isoformat()}")
report_lines.append("")
report_lines.append("## Scope")
report_lines.append("- Directed transaction graph analysis over all available timesteps.")
report_lines.append("- Three label groups: illicit (class1), licit (class2), unknown.")
report_lines.append("- Includes degree analysis, temporal components, clustering, distance, and assortativity.")
report_lines.append("")
report_lines.append("## Key Dataset Counts")
report_lines.append(f"- Total nodes analyzed: **{n_nodes:,}**")
report_lines.append(f"- Illicit nodes: **{n_illicit:,}**")
report_lines.append(f"- Licit nodes: **{n_licit:,}**")
report_lines.append(f"- Unknown-label nodes: **{n_unknown:,}**")
report_lines.append("")

report_lines.append("## Degree Statistics by Class (Total Degree)")
deg_tbl = pick_metric_table(class_summary, "degree")
report_lines.append(df_to_pipe_markdown(deg_tbl, decimals=4))
report_lines.append("")

report_lines.append("## Directed Behavior (Illicit Source Edges)")
report_lines.append(f"- P(illicit→illicit): **{safe_float(p_i_i.iloc[0] if len(p_i_i) else np.nan)}**")
report_lines.append(f"- P(illicit→licit): **{safe_float(p_i_l.iloc[0] if len(p_i_l) else np.nan)}**")
report_lines.append("")

report_lines.append("## Assortativity")
report_lines.append(f"- Label attribute assortativity: **{safe_float(assort_val.iloc[0] if len(assort_val) else np.nan)}**")
report_lines.append("")

report_lines.append("## Temporal Connectivity (Averages Across Timesteps)")
report_lines.append(f"- Avg weakly connected components: **{safe_float(avg_weak)}**")
report_lines.append(f"- Avg strongly connected components: **{safe_float(avg_strong)}**")
report_lines.append(f"- Avg giant-component share: **{safe_float(avg_giant_share)}**")
report_lines.append(f"- Avg shortest-path length in LCC (undirected): **{safe_float(avg_spl)}**")
report_lines.append(f"- Avg illicit↔illicit distance (LCC): **{safe_float(avg_illicit_dist)}**")
report_lines.append(f"- Avg random-pair distance (LCC): **{safe_float(avg_random_dist)}**")
report_lines.append("")

report_lines.append("## Average Clustering by Class")
report_lines.append(df_to_pipe_markdown(avg_clust_by_class, decimals=6))
report_lines.append("")

report_lines.append("## Output Artifacts")
report_lines.append("- Tables: outputs/graph_eda/tables")
report_lines.append("- Plots: outputs/graph_eda/plots")
report_lines.append("- Run log: outputs/graph_eda/logs/run_summary.txt")

report_text = "\n".join(report_lines)
report_path.write_text(report_text, encoding="utf-8")

print(f"Report written to: {report_path}")
display(Markdown(report_text))

Report written to: c:\Users\ashis\Desktop\Finance Projects\Deep Learning project 2026\outputs\graph_eda\report\checkpoint_graph_summary.md


# Graph Structural Analysis Summary

Generated on: 2026-03-02

## Scope
- Directed transaction graph analysis over all available timesteps.
- Three label groups: illicit (class1), licit (class2), unknown.
- Includes degree analysis, temporal components, clustering, distance, and assortativity.

## Key Dataset Counts
- Total nodes analyzed: **203,769**
- Illicit nodes: **4,545**
- Licit nodes: **42,019**
- Unknown-label nodes: **157,205**

## Degree Statistics by Class (Total Degree)
| class_group | min | max | mean | std | median | count |
| --- | --- | --- | --- | --- | --- | --- |
| illicit | 1.0000 | 177.0000 | 2.0117 | 7.1775 | 1.0000 | 4545.0000 |
| licit | 1.0000 | 473.0000 | 3.0952 | 7.7829 | 2.0000 | 42019.0000 |
| unknown | 1.0000 | 212.0000 | 2.0960 | 2.5281 | 2.0000 | 157205.0000 |

## Directed Behavior (Illicit Source Edges)
- P(illicit→illicit): **0.2961**
- P(illicit→licit): **0.2714**

## Assortativity
- Label attribute assortativity: **0.3403**

## Temporal Connectivity (Averages Across Timesteps)
- Avg weakly connected components: **1.0000**
- Avg strongly connected components: **4158.5510**
- Avg giant-component share: **1.0000**
- Avg shortest-path length in LCC (undirected): **15.4269**
- Avg illicit↔illicit distance (LCC): **9.1945**
- Avg random-pair distance (LCC): **15.2523**

## Average Clustering by Class
| class_group | local_clustering_directed | local_clustering_undirected |
| --- | --- | --- |
| illicit | 0.000178 | 0.000356 |
| licit | 0.004160 | 0.008320 |
| unknown | 0.007802 | 0.015604 |

## Output Artifacts
- Tables: outputs/graph_eda/tables
- Plots: outputs/graph_eda/plots
- Run log: outputs/graph_eda/logs/run_summary.txt